In [ ]:
# 导入四分类的数据
import import_ipynb
from MyPreprocess import *
from Extract_bispectrum import polycoherence,plot_polycoherence
import os

four_class_features = []
four_class_labels = []

for i, label in enumerate(['As', 'MR','MS','MVP']):
    folder_path = f'../HeartSound1/datasets/five_class/{label}/'
    for file in os.listdir(folder_path):
        audio_data, fs = librosa.load(os.path.join(folder_path, file), sr=None)
        audio_data = band_pass_filter(audio_data, 2, 25, 400, fs)
        down_sample_audio_data = resample_audio(audio_data, fs)
        down_sample_audio_data = add_noise1(down_sample_audio_data)
        down_sample_audio_data = add_noise2(down_sample_audio_data)
        down_sample_audio_data = normalize(down_sample_audio_data)
        ex_audio_data = down_sample_audio_data[:3000]
        freq1, freq2, bi_spectrum = polycoherence(ex_audio_data, nfft=1024, nperseg=256, noverlap=128, fs=2000, norm=None)
        bi_spectrum = np.array(abs(bi_spectrum))
        bi_spectrum = 255 * (bi_spectrum - np.min(bi_spectrum)) / (np.max(bi_spectrum) - np.min(bi_spectrum))
        bi_spectrum = bi_spectrum.reshape((256, 256, 1))
        
        # 添加到特征列表
        four_class_features.append(bi_spectrum)
        four_class_labels.append(i)  

In [ ]:
# 将提取到的特征转换为二进制文件 只需运行一次后续无需再运行
# import numpy as np
# np.save('four_class_features.npy', four_class_features)
# np.save('four_class_labels.npy', four_class_labels)

In [1]:
import os
import numpy as np
import librosa
from python_speech_features import mfcc
import matplotlib.pyplot as plt
import samplerate
from scipy import signal
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from imblearn.over_sampling import SVMSMOTE
import tensorflow.keras as keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix
import seaborn as sns
from keras.utils import to_categorical
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
# 读取二进制文件

four_class_features = np.load('four_class_features.npy')
four_class_labels = np.load('four_class_labels.npy')

In [4]:
import numpy as np
from sklearn.utils import shuffle

def custom_train_test_split(features, labels, test_size=0.3, random_state=42):
    """
    自定义数据集划分函数
    :param features: 特征数组 (num_samples, ...)
    :param labels: 标签数组 (num_samples,)
    :param test_size: 测试集占总数据的比例
    :param random_state: 随机种子，确保可重复性
    :return: train_features, test_features, train_label, test_label
    """
    # 确保输入是 NumPy 数组
    features = np.array(features)
    labels = np.array(labels)
    
    # 打乱数据和标签
    features, labels = shuffle(features, labels, random_state=random_state)
    
    # 计算训练集和测试集的大小
    num_samples = len(labels)
    test_size = int(num_samples * test_size)
    train_size = num_samples - test_size
    
    # 划分训练集和测试集
    train_features = features[:train_size]
    train_label = labels[:train_size]
    test_features = features[train_size:]
    test_label = labels[train_size:]
    
    return train_features, test_features, train_label, test_label

# 使用自定义函数划分数据集
train_features, test_features, train_label, test_label = custom_train_test_split(
    four_class_features, four_class_labels, test_size=0.3, random_state=42
)

# 打印划分后的数据集形状，确保划分正确
print("Train features shape:", train_features.shape)  
print("Train labels shape:", train_label.shape)       
print("Test features shape:", test_features.shape)    
print("Test labels shape:", test_label.shape)        

Train features shape: (560, 256, 256, 1)
Train labels shape: (560,)
Test features shape: (240, 256, 256, 1)
Test labels shape: (240,)


In [5]:
from keras.utils import to_categorical

# 将整数标签转换为 one-hot 编码
train_label = to_categorical(train_label, num_classes=4)
test_label = to_categorical(test_label, num_classes=4)

# 检查转换后的标签形状
print("Train labels shape:", train_label.shape)  
print("Test labels shape:", test_label.shape)    

Train labels shape: (560, 4)
Test labels shape: (240, 4)


In [ ]:
## 基准模型CNN

# from tensorflow import keras

# def build_model():
#     inputdata = keras.Input(shape=(256, 256, 1))
#     final = inputdata / 255.0
    

#     final = keras.layers.Conv2D(32, (3, 3), padding="same",activation='relu')(inputdata)
#     final = keras.layers.BatchNormalization()(final)
#     final = keras.layers.ReLU()(final)
#     final = keras.layers.MaxPooling2D((2, 2), strides=(2, 2))(final)
#     final = keras.layers.Dropout(0.3)(final)

#     final = keras.layers.Conv2D(16, (3, 3), padding="same",activation='relu')(final)
#     final = keras.layers.BatchNormalization()(final)
#     final = keras.layers.ReLU()(final)
#     final = keras.layers.MaxPooling2D((2, 2), strides=(2, 2))(final)
#     final = keras.layers.Dropout(0.3)(final)

#     final = keras.layers.Conv2D(8, (3, 3), padding="same",activation='relu')(final)
#     final = keras.layers.BatchNormalization()(final)
#     final = keras.layers.ReLU()(final)
#     final = keras.layers.MaxPooling2D((2, 2), strides=(2, 2))(final)
#     final = keras.layers.Dropout(0.3)(final)

#     final = keras.layers.Conv2D(16, (3, 3), padding="same",activation='relu')(final)
#     final = keras.layers.BatchNormalization()(final)
#     final = keras.layers.ReLU()(final)
#     final = keras.layers.Dropout(0.3)(final)

#     final = layers.Flatten()(final)
#     final = layers.Dense(128, activation='relu', kernel_regularizer=keras.regularizers.l2(0.001))(final)  # 添加 L2 正则化
#     final = layers.BatchNormalization()(final)
#     final = layers.Dropout(0.6)(final)  
#     final = layers.Dense(4)(final)
#     final = layers.Softmax()(final)


#     model = keras.Model(inputs=inputdata, outputs=final)
#     optimizer = keras.optimizers.Adam(
#         learning_rate=0.001, beta_1=0.9, beta_2=0.999, epsilon=None)
#     model.compile(loss='categorical_crossentropy',
#                   optimizer=optimizer,
#                   metrics=['accuracy'])
#     # print(model.summary())
#     return model

# model = build_model()

In [ ]:
# import tensorflow as tf
# from tensorflow import keras
# from tensorflow.keras import layers

#  多尺度卷积（MS） + Squeeze-and-Excitation 模块

# def ms_se_block(x, filters):
#     # 多尺度卷积分支
#     b1 = layers.Conv2D(filters, (1, 1), padding="same", activation=None)(x)
#     b3 = layers.Conv2D(filters, (3, 3), padding="same", activation=None)(x)
#     b5 = layers.Conv2D(filters, (5, 5), padding="same", activation=None)(x)
#     out = layers.Concatenate()([b1, b3, b5])
#     out = layers.BatchNormalization()(out)
#     out = layers.ReLU()(out)

#     # Squeeze-and-Excitation
#     se = layers.GlobalAveragePooling2D()(out)
#     se = layers.Dense(filters * 3 // 16, activation='relu')(se)
#     se = layers.Dense(filters * 3, activation='sigmoid')(se)
#     se = layers.Reshape((1, 1, filters * 3))(se)
#     out = layers.Multiply()([out, se])
#     return out

# # 构建完整模型

# def build_model(input_shape=(256, 256, 1), num_classes=4):
#     inputs = keras.Input(shape=input_shape)

#     # 初始卷积层
#     x = layers.Conv2D(32, (3, 3), padding="same")(inputs)
#     x = layers.BatchNormalization()(x)
#     x = layers.ReLU()(x)
#     x = layers.MaxPooling2D((2, 2))(x)
#     x = layers.Dropout(0.3)(x)

#     # 第一个多尺度SE模块
#     x = ms_se_block(x, filters=8)
#     x = layers.MaxPooling2D((2, 2))(x)
#     x = layers.Dropout(0.3)(x)

#     # 第二组卷积
#     x = layers.Conv2D(16, (3, 3), padding="same")(x)
#     x = layers.BatchNormalization()(x)
#     x = layers.ReLU()(x)
#     x = layers.MaxPooling2D((2, 2))(x)
#     x = layers.Dropout(0.3)(x)

#     # 第三个多尺度SE模块
#     x = ms_se_block(x, filters=16)
#     x = layers.MaxPooling2D((2, 2))(x)
#     x = layers.Dropout(0.3)(x)

#     # 全局特征汇聚与分类头
#     x = layers.Flatten()(x)
#     x = layers.Dense(64, activation='relu', kernel_regularizer=keras.regularizers.l2(0.0005))(x)
#     x = layers.BatchNormalization()(x)
#     x = layers.Dropout(0.5)(x)
#     x = layers.Dense(num_classes)(x)
#     outputs = layers.Softmax()(x)

#     model = keras.Model(inputs, outputs, name="ms_se_cnn")

#     # 编译模型
#     optimizer = keras.optimizers.Adam(learning_rate=1e-3)
#     model.compile(
#         loss='categorical_crossentropy',
#         optimizer=optimizer,
#         metrics=['accuracy']
#     )
#     return model
# model = build_model()


In [ ]:
print("Model input shape:", model.input_shape)
print("Model output shape:", model.output_shape)

# 编译模型
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# 添加调试回调
from keras.callbacks import LambdaCallback

def log_debug(epoch, logs):
    print(f"Epoch {epoch + 1}, Logs: {logs}")

debug_callback = LambdaCallback(on_epoch_end=log_debug)

# 训练模型
num_epochs = 100
history = model.fit(train_features, train_label,
                        epochs=num_epochs,
                        batch_size=32,
                        validation_split=0.2,
                        verbose=1)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# 设置Seaborn的主题和调色板
sns.set(style="white", palette="bright")  

def cal_acc(label, prediction):
    num = 0
    N = len(label)
    pred_class = np.argmax(prediction, axis=1)  
    for i in range(len(label)):
        if np.argmax(label[i]) == pred_class[i]:
            num += 1
    return num / N

# 绘制损失曲线
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
sns.lineplot(x=range(1, len(history.history['loss']) + 1), y=history.history['loss'], label='Training Loss')
sns.lineplot(x=range(1, len(history.history['val_loss']) + 1), y=history.history['val_loss'], label='Validation Loss')
plt.title('Loss Curves')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# 绘制准确率曲线
plt.subplot(1, 2, 2)
sns.lineplot(x=range(1, len(history.history['accuracy']) + 1), y=history.history['accuracy'], label='Training Accuracy')
sns.lineplot(x=range(1, len(history.history['val_accuracy']) + 1), y=history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy Curves')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

# 预测测试集
test_predictions = model.predict(test_features)
acc = cal_acc(test_label, test_predictions)
print("测试集准确率为：", acc)

# 计算混淆矩阵
cm = confusion_matrix(np.argmax(test_label, axis=1), np.argmax(test_predictions, axis=1))
print("Confusion Matrix:")
print(cm)

# 绘制混淆矩阵
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["AS", "MR", "MS", "MVP"])
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.show()